# F21MP — Full Pipeline

Runs all phases end-to-end. Use this to re-run everything from scratch or check a specific phase.
Individual phase notebooks in `notebooks/phase*/` are still the main working notebooks.


Run from the project root. The root-finding code below handles `sys.path` automatically.


## Setup


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('Cannot find F21MP project root.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)
print('Python:', sys.executable)


Project root: /Users/sagarmamindla/Desktop/F21MP
Python: /Users/sagarmamindla/Desktop/F21MP/.venv/bin/python


In [2]:
# Optional quick dependency check
import importlib
for pkg in ['pandas', 'numpy', 'sklearn', 'xgboost', 'matplotlib', 'joblib']:
    mod = importlib.import_module(pkg)
    ver = getattr(mod, '__version__', 'unknown')
    print(f'{pkg}: {ver}')


pandas: 3.0.3
numpy: 2.4.6
sklearn: 1.9.0
xgboost: 3.2.0
matplotlib: 3.10.9
joblib: 1.5.3


---
## Phase 1 — Training


In [3]:
from src.phase1_training.models.train import train_phase1

phase1 = train_phase1(test_size=0.2, random_state=42, save_outputs=True, verbose=True)


Total samples: 15501
Train: 13031 | Test: 2470
Train pairs: 6691
Test pairs:  1673
Train CBLabel counts:
CBLabel
0    12930
1      101
Name: count, dtype: int64
scale_pos_weight: 128.02

=== Baseline A1 (text-only) ===
Accuracy:  0.9660
Precision: 0.0000
Recall:    0.0000
F1:        0.0000
ROC-AUC:   0.5095
Confusion matrix: [[2386, 59], [25, 0]]

=== Full model A4 (text + peerness + repetition) ===
Accuracy:  0.9996
Precision: 0.9615
Recall:    1.0000
F1:        0.9804
ROC-AUC:   1.0000
Confusion matrix: [[2444, 1], [0, 25]]

Models saved to: /Users/sagarmamindla/Desktop/F21MP/outputs/models
Metrics saved to: /Users/sagarmamindla/Desktop/F21MP/outputs/results/phase1_test_metrics.json


In [4]:
phase1_summary = pd.DataFrame([
    {
        'model': 'A1_text_only',
        **{k: phase1['baseline_metrics'][k] for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    },
    {
        'model': 'A4_full',
        **{k: phase1['full_metrics'][k] for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    },
])
phase1_summary


,model,accuracy,precision,recall,f1,roc_auc
0,A1_text_only,0.965992,0.000000,0.0,0.000000,0.509497
1,A4_full,0.999595,0.961538,1.0,0.980392,0.999984


---
## Phase 2 — Cross-dataset validation


In [5]:
from src.phase2_validation.evaluation.validation import (
    run_validation,
    sweep_thresholds,
    run_domain_adaptive_finetune,
    run_embedding_extension,
)

val_text = run_validation(strategy='text_only', threshold=0.5, save_outputs=True)
val_full = run_validation(strategy='zero_shot_full', threshold=0.5, save_outputs=True)

display(val_text['summary'])
display(val_full['summary'])


,subset,samples,positive,accuracy,precision,recall,f1,roc_auc
0,twitter_parsed_dataset.csv,16848,5347,0.684651,0.610390,0.017580,0.034176,0.512294
1,twitter_racism_parsed_dataset.csv,13471,1970,0.851088,0.285714,0.012183,0.023369,0.492103
2,twitter_sexism_parsed_dataset.csv,14878,3377,0.773693,0.538462,0.020728,0.039920,0.524072
3,youtube_parsed_dataset.csv,3464,417,0.877309,0.100000,0.002398,0.004684,0.375216


,subset,samples,positive,accuracy,precision,recall,f1,roc_auc
0,twitter_parsed_dataset.csv,16848,5347,0.682633,0.0,0.0,0.0,0.493973
1,twitter_racism_parsed_dataset.csv,13471,1970,0.853760,0.0,0.0,0.0,0.406171
2,twitter_sexism_parsed_dataset.csv,14878,3377,0.773021,0.0,0.0,0.0,0.545192
3,youtube_parsed_dataset.csv,3464,417,0.879619,0.0,0.0,0.0,0.599143


In [6]:
threshold_sweep = sweep_thresholds(
    strategy='text_only',
    thresholds=[0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.50],
    save_outputs=True,
)
threshold_sweep.head(12)


,subset,samples,positive,accuracy,precision,recall,f1,roc_auc,threshold
0,twitter_parsed_dataset.csv,16848,5347,0.607550,0.325806,0.221246,0.263533,0.512294,0.10
1,twitter_racism_parsed_dataset.csv,13471,1970,0.699280,0.130373,0.186294,0.153396,0.492103,0.10
2,twitter_sexism_parsed_dataset.csv,14878,3377,0.663328,0.250000,0.241635,0.245746,0.524072,0.10
3,youtube_parsed_dataset.csv,3464,417,0.799365,0.073620,0.057554,0.064603,0.375216,0.10
4,twitter_parsed_dataset.csv,16848,5347,0.623160,0.305512,0.147185,0.198662,0.512294,0.15
5,twitter_racism_parsed_dataset.csv,13471,1970,0.737213,0.109064,0.111168,0.110106,0.492103,0.15
6,twitter_sexism_parsed_dataset.csv,14878,3377,0.690953,0.240984,0.168197,0.198116,0.524072,0.15
7,youtube_parsed_dataset.csv,3464,417,0.820150,0.063559,0.035971,0.045942,0.375216,0.15
8,twitter_parsed_dataset.csv,16848,5347,0.633013,0.284979,0.103610,0.151968,0.512294,0.20
9,twitter_racism_parsed_dataset.csv,13471,1970,0.761785,0.097988,0.076650,0.086015,0.492103,0.20


In [7]:
best_thresholds = (
    threshold_sweep.sort_values(['subset', 'f1'], ascending=[True, False])
    .groupby('subset', as_index=False)
    .first()[['subset', 'threshold', 'f1', 'precision', 'recall']]
)
best_thresholds


,subset,threshold,f1,precision,recall
0,twitter_parsed_dataset.csv,0.1,0.263533,0.325806,0.221246
1,twitter_racism_parsed_dataset.csv,0.1,0.153396,0.130373,0.186294
2,twitter_sexism_parsed_dataset.csv,0.1,0.245746,0.250000,0.241635
3,youtube_parsed_dataset.csv,0.1,0.064603,0.073620,0.057554


In [8]:
adapt = run_domain_adaptive_finetune(
    subset_name='twitter_parsed_dataset.csv',
    adapt_fraction=0.20,
    random_state=42,
    save_outputs=True,
)
embed = run_embedding_extension(
    subset_name='twitter_parsed_dataset.csv',
    adapt_fraction=0.20,
    random_state=42,
    save_outputs=True,
)

phase2_ext = pd.DataFrame([
    {'experiment': 'domain_adaptive_finetune', 'threshold': adapt['threshold_selected_on_calib'], **adapt['test_metrics']},
    {'experiment': 'embedding_extension', 'threshold': embed['threshold_selected_on_calib'], **embed['test_metrics']},
])
phase2_ext


/Users/sagarmamindla/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,experiment,threshold,accuracy,precision,recall,f1,confusion_matrix,support,roc_auc,classification_report
0,domain_adaptive_finetune,0.4,0.752504,0.597677,0.673679,0.633407,"[[7261, 1940], [1396, 2882]]","{'n_samples': 13479, 'n_positive': 4278, 'n_ne...",0.802811,"{'0': {'precision': 0.8387432135843825, 'recal..."
1,embedding_extension,0.3,0.752875,0.593190,0.704535,0.644086,"[[7134, 2067], [1264, 3014]]","{'n_samples': 13479, 'n_positive': 4278, 'n_ne...",0.817183,"{'0': {'precision': 0.8494879733269826, 'recal..."


---
## Phase 2 — Robustness and error analysis


In [9]:
from src.phase2_validation.evaluation.validation import (
    run_adaptation_robustness,
    run_misclassification_analysis,
)

robust = run_adaptation_robustness(
    subset_name='twitter_parsed_dataset.csv',
    adapt_fraction=0.20,
    seeds=(13, 42, 77),
    save_outputs=True,
)

display(robust['per_seed'])


/Users/sagarmamindla/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/sagarmamindla/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/sagarmamindla/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,seed,experiment,f1,roc_auc,threshold
0,13,domain_adaptive_finetune,0.651141,0.816152,0.45
1,42,domain_adaptive_finetune,0.633407,0.802811,0.40
2,77,domain_adaptive_finetune,0.628615,0.805724,0.55
3,13,embedding_extension,0.646124,0.817105,0.30
4,42,embedding_extension,0.644086,0.817183,0.30
5,77,embedding_extension,0.638536,0.810153,0.40


In [10]:
robust['summary']


,experiment,runs,f1_mean,f1_std,f1_min,f1_max,f1_ci95_low,f1_ci95_high,roc_auc_mean,threshold_mean
0,domain_adaptive_finetune,3,0.637721,0.011867,0.628615,0.651141,0.628615,0.651141,0.808229,0.466667
1,embedding_extension,3,0.642915,0.003927,0.638536,0.646124,0.638536,0.646124,0.814814,0.333333


In [11]:
err = run_misclassification_analysis(
    subset_name='twitter_parsed_dataset.csv',
    experiment='embedding_extension',
    adapt_fraction=0.20,
    random_state=42,
    top_n=20,
    save_outputs=True,
)
print('Selected threshold:', round(err['threshold'], 4))
print('Total test errors:', err['n_errors_total'])
err['errors'].head(20)


Selected threshold: 0.3
Total test errors: 3331


/Users/sagarmamindla/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,text,y_true,y_pred,y_proba,error_type,confidence_distance
0,RT @lennyfinnegan2 I'm not sexist but http://t...,0,1,0.999977,FP,0.699977
1,RT @CarSalesBossMan: I'm not sexist but http:/...,0,1,0.999977,FP,0.699977
2,RT @James32Webb: Football logic for girls #not...,0,1,0.999067,FP,0.699067
3,RT @DrDayNothing: Never Ending Horror of Scum ...,0,1,0.997751,FP,0.697751
4,RT @Vrais66: PROPAGEZ Islam = Lapidation Amput...,0,1,0.997490,FP,0.697490
5,"@a_little_opus Treating women like trophies, t...",0,1,0.995852,FP,0.695852
6,@MadDr11 @MaxBlumenthal No they don't. 40% of ...,0,1,0.995242,FP,0.695242
7,"RT @BrettJGlasco: I'm NOT sexist, but this is ...",0,1,0.995171,FP,0.695171
8,"RT @BeRh00M: I'm not sexist, but I posted this...",0,1,0.994836,FP,0.694836
9,@AndyG244 @Thearetical @Demon31G They are lyin...,0,1,0.994533,FP,0.694533


---
## Phase 3 — Ablation and hate speech benchmark


In [12]:
from src.phase3_ablation_benchmark.evaluation.ablation import run_ablation, run_hate_speech_benchmark

ablation = run_ablation(test_size=0.2, random_state=42, save_outputs=True)
hate = run_hate_speech_benchmark(test_size=0.2, random_state=42, save_outputs=True)

print('Ablation summary')
display(ablation['summary'])
print('Hate-speech benchmark summary')
display(hate['summary'])


/Users/sagarmamindla/Desktop/F21MP/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Ablation summary


,experiment,accuracy,precision,recall,f1,roc_auc
0,A1_text_only,0.965992,0.000000,0.0,0.000000,0.509497
1,A2_text_plus_peerness,0.991093,0.540541,0.8,0.645161,0.995419
2,A3_text_plus_repetition,0.943320,0.040000,0.2,0.066667,0.686020
3,A4_full,0.999595,0.961538,1.0,0.980392,0.999984
4,A5_full_plus_demographics,0.999595,0.961538,1.0,0.980392,0.999984


Hate-speech benchmark summary


,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,n_samples
0,tfidf_logreg,0.892677,0.807038,0.627424,0.656370,0.874876,4957
1,tfidf_xgboost,0.910026,0.775098,0.716196,0.724195,0.901194,4957


---
## Phase 4 — OLID benchmark (optional)


In [13]:
RUN_PHASE4 = True  # set False to skip this optional phase

if RUN_PHASE4:
    from src.phase4_olid.data.load_olid import run_olid_benchmark
    olid = run_olid_benchmark(save_outputs=True)
    display(olid['summary'])
else:
    print('Phase 4 skipped.')


,model,accuracy,precision,recall,f1,n_samples,n_positive,roc_auc
0,zero_shot_phase1_a1,0.716279,0.000000,0.000000,0.000000,860,240,0.501099
1,olid_in_domain_text_only,0.791860,0.640553,0.579167,0.608315,860,240,0.793219


---
## Combined results summary


In [14]:
rows = []

if 'phase1_summary' in globals():
    for _, r in phase1_summary.iterrows():
        rows.append({'phase': 'phase1', 'experiment': r['model'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

if 'phase2_ext' in globals():
    for _, r in phase2_ext.iterrows():
        rows.append({'phase': 'phase2_extension', 'experiment': r['experiment'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

if 'ablation' in globals():
    for _, r in ablation['summary'].iterrows():
        rows.append({'phase': 'phase3_ablation', 'experiment': r['experiment'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

if 'hate' in globals():
    for _, r in hate['summary'].iterrows():
        rows.append({'phase': 'phase3_hate_speech', 'experiment': r['model'], 'f1': r.get('macro_f1'), 'roc_auc': np.nan})

if 'olid' in globals() and RUN_PHASE4:
    for _, r in olid['summary'].iterrows():
        rows.append({'phase': 'phase4_olid', 'experiment': r['model'], 'f1': r.get('f1'), 'roc_auc': r.get('roc_auc')})

final_summary = pd.DataFrame(rows)
final_summary


,phase,experiment,f1,roc_auc
0,phase1,A1_text_only,0.000000,0.509497
1,phase1,A4_full,0.980392,0.999984
2,phase2_extension,domain_adaptive_finetune,0.633407,0.802811
3,phase2_extension,embedding_extension,0.644086,0.817183
4,phase3_ablation,A1_text_only,0.000000,0.509497
5,phase3_ablation,A2_text_plus_peerness,0.645161,0.995419
6,phase3_ablation,A3_text_plus_repetition,0.066667,0.686020
7,phase3_ablation,A4_full,0.980392,0.999984
8,phase3_ablation,A5_full_plus_demographics,0.980392,0.999984
9,phase3_hate_speech,tfidf_logreg,0.656370,NaN
